### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from IPython.display import clear_output
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate
import tessellate

### Options

In [ ]:
fov = 120.0
radius = fov/2.0

### Generate "Eigen" Asterisms

In [ ]:
r, theta = tessellate.generate_symmetric_polar_grid(
    n_rings = 6,
    n_first_ring=1,
    r_min=10,
    r_max=60,
    delta_n=1,
    return_polar=True
)
print(f"Number of points in circle: {len(r)}")
tessellate.plot_polar_grid(r, theta, radius, title=f"Symmetrical Grid with {len(r)} points")
#point_headers = ["Point Index", "Radius", "Angle (°)"]
#display(tabulate(list(zip(np.arange(len(r)), r, np.rad2deg(theta))), headers=point_headers, tablefmt='html'))

In [ ]:
asterisms, stats = tessellate.generate_asterisms(r, theta, max_incentre_distance=2.2, return_stats=True)
print(f"Number of possible asterisms: {stats.num_triplets}")
print(f"Cut asterisms with multiple stars at a single point: {stats.num_non_repeat_triplets}")
print(f"Number of asterisms that are not degenerate: {stats.num_non_repeat_triplets - stats.num_degenerate}")
print(f"Number of asterisms with centroid within max radius ({stats.max_incentre_distance}): {stats.num_non_repeat_triplets - stats.num_degenerate - stats.num_not_centered}")
print(f"Number of geometrically distinct asterisms: {stats.num_asterisms}")

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='polar')
ax.set_title(f"{len(asterisms)} Geometrically Distinct Asterisms")
ax.set_rticks([10, 20, 30, 40, 50])
ax.grid(True)
ax.set_rlim(0, radius)

for asterism in asterisms:
    plot_r = list(asterism.r) + [asterism.r[0]]
    plot_theta = list(asterism.theta) + [asterism.theta[0]]
    ax.plot(plot_theta, plot_r, alpha=0.7)
    clear_output(wait=True)
    display(fig)
    time.sleep(0.5)

plt.close(fig)

In [ ]:
def build_asterism_table_data(asterisms, degrees=True):
    """
    Builds the raw data and headers for a table of asterisms with star indices, r, and theta.
    """
    table_data = []
    for i, asterism in enumerate(asterisms):
        row = []
        row.append(i+1)
        row.append(round(asterism.center_r, 1))
        row.append(round(np.rad2deg(asterism.center_theta), 1) if degrees else round(asterism.center_theta, 1))
        row.append(round(asterism.score, 3))
        row.append(round(asterism.scale, 1))
        row.append(round(asterism.area, 1))
        for j in range(len(asterism.r)):
            row.append(asterism.r[j])
            angle = np.rad2deg(asterism.theta[j]) if degrees else asterism.theta[j]
            row.append(round(angle, 1))
        table_data.append(row)

    headers = ["Asterism", "Center r", "Center θ", "Score", "Scale", "Area"]
    for i in range(1, len(asterisms[0].r)+1):
        headers += [f"r{i}", f"θ{i} (°)" if degrees else f"θ{i} (rad)"]

    return table_data, headers

table_data, headers = build_asterism_table_data(asterisms)
display(tabulate(table_data, headers=headers, tablefmt='html'))

### Asterism Scale

In [ ]:
asterism_id = 10

asterism = asterisms[asterism_id]

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='polar')
ax.set_rticks([10, 20, 30, 40, 50])
ax.grid(True)
ax.set_rlim(0, radius)
plot_r = list(asterism.r) + [asterism.r[0]]
plot_theta = list(asterism.theta) + [asterism.theta[0]]
ax.plot(plot_theta, plot_r, c='blue', alpha=0.7)
circle = plt.Circle((asterism.center_theta, asterism.center_r), 1, transform=ax.transData._b, color='blue', alpha=0.7)
ax.add_artist(circle)
circle = plt.Circle((asterism.center_theta, asterism.center_r), asterism.scale/2, transform=ax.transData._b, color='red', alpha=0.35)
ax.add_artist(circle)
plt.show()

### Asterism Statistics

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(np.sort([asterism.scale for asterism in asterisms]), marker='o', linestyle='-', color='g', label='Asterism Scales')
plt.xlabel('Asterism Count')
plt.ylabel('Scale [arcsec]')
plt.title('Asterism Scale')
plt.axhline(y=fov, color='k', linestyle='--')
plt.text(0, fov, 'Max Scale', color='k', ha='left', va='bottom')
plt.ylim(0, fov*1.05)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
max_area = 3 * np.sqrt(3) / 16 * fov**2
plt.figure(figsize=(10, 6))
plt.plot(np.sort([asterism.area for asterism in asterisms]), marker='o', linestyle='-', color='r', label='Asterism Areas')
plt.xlabel('Asterism Count')
plt.ylabel('Area [arcsec²]')
plt.title('Asterism Area')
plt.axhline(y=max_area, color='k', linestyle='--')
plt.text(0, max_area, 'Max Area', color='k', ha='left', va='bottom')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot([asterism.score for asterism in asterisms], marker='o', linestyle='-', color='b', label='Asterism Scores')
plt.xlabel('Asterism Index')
plt.ylabel('Score')
plt.title('Asterism Score')
plt.axhline(y=1.0, color='k', linestyle='--')
plt.text(50, 1.0, 'Equilateral Triangle', color='k', ha='right', va='bottom')
plt.grid(True)
plt.legend()
plt.show()